In [1]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2




r_seed = 42

from pathlib import Path
WORKDIR = str(Path.cwd().parent) + '/'  # raiz do repositório
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/logs/"+data_hora

In [2]:

# Assuming your data is saved as 'data.csv'
# Load the data from the CSV file using pandas
df = pd.read_csv(WORKDIR + 'Data/processed/df_X_train.csv')

# Extract the actual 3D array from the DataFrame


X_train = df.to_numpy().reshape(125532, 200, 27)
#data = df.to_numpy().reshape(2,3,4)

print("3D array loaded and reshaped back to its original dimensions.")
print(X_train.shape)  # Output: (2, 3, 4)

3D array loaded and reshaped back to its original dimensions.
(125532, 200, 27)


In [3]:
# Assuming your data is saved as 'data.csv'
# Load the data from the CSV file using pandas
df_xval = pd.read_csv(WORKDIR + 'Data/processed/df_X_validation.csv')

# Extract the actual 3D array from the DataFrame


X_validation = df_xval.to_numpy().reshape(31453, 200, 27)
#data = df.to_numpy().reshape(2,3,4)

print("3D array loaded and reshaped back to its original dimensions.")
print(X_validation.shape)  # Output: (2, 3, 4)

3D array loaded and reshaped back to its original dimensions.
(31453, 200, 27)


In [4]:
# Assuming your data is saved as 'data.csv'
# Load the data from the CSV file using pandas
df_ytrain = pd.read_csv(WORKDIR + 'Data/processed/df_y_train.csv')

# Extract the actual 3D array from the DataFrame


y_train = df_ytrain.to_numpy().reshape(125532, 18)
#data = df.to_numpy().reshape(2,3,4)

print("3D array loaded and reshaped back to its original dimensions.")
print(y_train.shape)  # Output: (2, 3, 4)

3D array loaded and reshaped back to its original dimensions.
(125532, 18)


In [5]:
# Assuming your data is saved as 'data.csv'
# Load the data from the CSV file using pandas
df_yval = pd.read_csv(WORKDIR + 'Data/processed/df_y_validation.csv')

# Extract the actual 3D array from the DataFrame


y_validation = df_yval.to_numpy().reshape(31453, 18)
#data = df.to_numpy().reshape(2,3,4)

print("3D array loaded and reshaped back to its original dimensions.")
print(y_validation.shape)  # Output: (2, 3, 4)

3D array loaded and reshaped back to its original dimensions.
(31453, 18)


In [6]:
print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape)

(125532, 200, 27)
(125532, 18)
(31453, 200, 27)
(31453, 18)


In [7]:
def shuffle_data(variable_data,label_data):
    perm_indices = np.random.permutation(variable_data.shape[0])
    # Shuffle both arrays using the same permutation indices
    shuffled_variable_data = variable_data[perm_indices]
    shuffled_label_data = label_data[perm_indices]
    return (shuffled_variable_data,shuffled_label_data)

In [8]:
#X_train,y_train = shuffle_data(X_train,y_train)
#X_validation,y_validation = shuffle_data(X_validation,y_validation)

In [9]:
""" print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape) """

' print(X_train.shape)\nprint(y_train.shape)\nprint(X_validation.shape)\nprint(y_validation.shape) '

In [10]:
#from tensorflow.keras import regularizers

In [15]:
def cria_LSTM(hp):
  model = tf.keras.Sequential()  
  model.add(tf.keras.layers.LSTM(hp.Int("input_units",min_value=32,max_value=256,step=32), input_shape=(200,27), return_sequences=False)) 
  model.add(tf.keras.layers.BatchNormalization())   
  model.add(tf.keras.layers.Dropout(rate=hp.Float(
                'dropout_1',
                min_value=0.25,
                max_value=0.5,
                default=0.25,
                step=0.05,
            )))
  model.add(tf.keras.layers.Dense(hp.Int("input_units_relu",min_value=32,max_value=256,step=32), activation='relu'))  
  model.add(tf.keras.layers.Dropout(rate=hp.Float(
                'dropout_1',
                min_value=0.25,
                max_value=0.5,
                default=0.25,
                step=0.05,
            )))
  model.add(tf.keras.layers.Dense(units=18, activation='softmax'))
  model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(hp.Float(
                    'learning_rate',
                    min_value=1e-4,
                    max_value=1e-2,
                    sampling='LOG',
                    default=1e-3
                )), metrics=['accuracy'])
  return model


In [16]:
tuner = RandomSearch(
    cria_LSTM,
    objective = 'val_accuracy', 
    seed=42,
    max_trials = 10,
    overwrite=True,        
    executions_per_trial = 1,    
    directory = LOG_DIR,
    project_name="lstm_acel_giro_multiclass")

In [17]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=7,
    min_delta = 0,
    verbose=1,
    restore_best_weights=True)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    verbose=1)

tensorboard = tf.keras.callbacks.TensorBoard(LOG_DIR)

my_callbacks = [early_stopping,reduce_lr,tensorboard]  


In [18]:
tuner.search(x=X_train,
            y=y_train,
            epochs=50,
            validation_data=(X_validation,y_validation), callbacks=my_callbacks,batch_size=16, use_multiprocessing=True)


Search: Running Trial #1

Value             |Best Value So Far |Hyperparameter
192               |192               |input_units
0.25              |0.25              |dropout_1
128               |128               |input_units_relu
0.00034974        |0.00034974        |learning_rate

Epoch 1/50
7846/7846 [==============================] - 1214s 154ms/step - loss: 1.2636 - accuracy: 0.5882 - val_loss: 1.1812 - val_accuracy: 0.6534 - lr: 3.4974e-04
Epoch 2/50
7846/7846 [==============================] - ETA: 0s - loss: 0.7539 - accuracy: 0.7481
Epoch 2: ReduceLROnPlateau reducing learning rate to 0.0001748722279444337.
7846/7846 [==============================] - 1192s 152ms/step - loss: 0.7539 - accuracy: 0.7481 - val_loss: 1.2667 - val_accuracy: 0.6941 - lr: 3.4974e-04
Epoch 3/50
 891/7846 [==>...........................] - ETA: 16:10 - loss: 0.5602 - accuracy: 0.8094

In [ ]:
tuner.results_summary()

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
best_model.summary()

In [ ]:
loss, accuracy = best_model.evaluate(X_validation, y_validation)

In [ ]:
predictions = best_model.predict(X_validation)

In [ ]:
print(y_validation)

In [ ]:
print(y_validation.shape)

In [ ]:
y_validation_inverse = np.argmax(y_validation, axis=1)

In [ ]:
print(y_validation_inverse)

In [ ]:
print(len(y_validation_inverse))

In [ ]:
print(predictions)

In [ ]:
classes_x=np.argmax(predictions,axis=1)

In [ ]:
print(classes_x)

In [ ]:
print(len(classes_x))

In [ ]:
print(classes_x.shape)

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report
cm = confusion_matrix(y_validation_inverse, classes_x)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

In [ ]:
disp.plot()

In [ ]:
print(classification_report(y_validation_inverse, classes_x))

In [ ]:
best_model.save(WORKDIR+"/Model/best_models_saved")

In [ ]:
import pandas as pd

# Create a DataFrame, optionally adding column names
df_X_train = pd.DataFrame(X_train.reshape(-1, X_train.shape[-1]))
df_X_train.columns = ["1", "2", "3", "4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19",
                           "20","21","22","23","24","25","26","27"]  # Optional: Add column names

# Save the DataFrame to a CSV file
df_X_train.to_csv(WORKDIR + 'Data/processed/df_X_train.csv', index=False)

print("3D array saved to df_X_train.csv")

In [ ]:
import pandas as pd

# Create a DataFrame, optionally adding column names
df_X_validation = pd.DataFrame(X_validation.reshape(-1, X_validation.shape[-1]))
df_X_validation.columns = ["1", "2", "3", "4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19",
                           "20","21","22","23","24","25","26","27"]  # Optional: Add column names

# Save the DataFrame to a CSV file
df_X_validation.to_csv(WORKDIR + 'Data/processed/df_X_validation.csv', index=False)

print("3D array saved to data.csv")

In [ ]:
import pandas as pd

# Create a DataFrame, optionally adding column names
df_X_train = pd.DataFrame(X_train.reshape(-1, X_train.shape[-1]))
df_X_train.columns = ["1", "2", "3", "4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19",
                           "20","21","22","23","24","25","26","27"]  # Optional: Add column names

# Save the DataFrame to a CSV file
df_X_train.to_csv(WORKDIR + 'Data/processed/df_X_train.csv', index=False)

print("3D array saved to data.csv")

In [ ]:
import pandas as pd

# Create a DataFrame, optionally adding column names
df_y_train = pd.DataFrame(y_train.reshape(-1, y_train.shape[-1]))
df_y_train.columns = ["1", "2", "3", "4","5","6","7","8","9","10","11","12","13","14","15","16","17","18"]  # Optional: Add column names

# Save the DataFrame to a CSV file
df_y_train.to_csv(WORKDIR + 'Data/processed/df_y_train.csv', index=False)

print("3D array saved to df_y_train.csv")

In [ ]:
import pandas as pd

# Create a DataFrame, optionally adding column names
df_y_validation = pd.DataFrame(y_validation.reshape(-1, y_validation.shape[-1]))
df_y_validation.columns = ["1", "2", "3", "4","5","6","7","8","9","10","11","12","13","14","15","16","17","18"]  # Optional: Add column names

# Save the DataFrame to a CSV file
df_y_validation.to_csv(WORKDIR + 'Data/processed/df_y_validation.csv', index=False)

print("3D array saved to y_validation.csv")